
# SKIRTOR AGN torus: inclination-dependent obscuration

Demonstrate how the SKIRTOR clumpy radiative-transfer torus (Stalevski+2016)
reprocesses the hot accretion disc as a function of viewing angle.

**Physical picture:**

- **Face-on (0°, cos_inc=1)**: The hot inner disc dominates. The torus is
  seen from above; torus emission is minimal and the UV/optical big blue bump
  fully visible.

- **Edge-on (90°, cos_inc=0)**: Heavy obscuration. The torus is seen edge-on;
  the disc is completely hidden and the re-radiated MIR/FIR torus emission
  dominates the SED.

The SED transitions continuously as inclination increases, illustrating
the unified AGN model: Seyfert 1s and 2s are the same object viewed at
different angles.

**Reference:** Stalevski et al. 2016, MNRAS, clumpy 3D radiative-transfer
modeling of dusty obscuring structures around AGN.


In [ ]:
import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18

# Inclination angles to sweep: 0° (face-on, cos_inc=1) to 90° (edge-on, cos_inc=0)
inclination_deg = np.array([0.0, 15.0, 30.0, 45.0, 60.0, 75.0, 90.0])
cos_inc_values = np.cos(np.radians(inclination_deg))

# Minimal SED model: just the AGN disc + torus, no starburst dust
SFH = {"type": "const", "*": tengri.FIXED, "log_sfr": -10.0}
DUST = {"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0}

ssp = tengri.load_ssp()

# Build model once, then modify inclination for each prediction
model = tengri.SEDModel.build(
    ssp,
    sfh=SFH,
    dust=DUST,
    agn={
        "disc": {"type": "multicolor", "*": tengri.FIXED},
        "torus": {"type": "skirtor", "*": tengri.FIXED},
        "*": tengri.FIXED,
        "log_lbol": 12.0,
        "frac": 1.0,
    },
    redshift=tengri.Fixed(0.0),
)

# Sample baseline params
baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))

fig, ax = plt.subplots(figsize=(8.5, 5.5))

# Colormap for inclination angle
colors = plt.cm.viridis(np.linspace(0.0, 1.0, len(inclination_deg)))

for cos_inc, inc_deg, color in zip(cos_inc_values, inclination_deg, colors):
    # Modify inclination parameter for this prediction
    params = {**baseline, "agn_cos_inc": np.float64(cos_inc)}
    out = model.predict_rest_sed(params)
    wave = np.asarray(out.wavelength)
    sed = np.asarray(out.sed)
    nu_l_nu = C_AA_PER_S / wave * sed

    ax.loglog(
        wave,
        nu_l_nu,
        color=color,
        lw=1.6,
        label=f"{inc_deg:5.1f}°",
        alpha=0.85,
    )

# Axis labels and limits
ax.set(
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]",
    xlim=(1e3, 1e6),
    ylim=(1e43, 1e47),
)

# Legend: inclination angles
legend = ax.legend(
    title="Inclination angle",
    frameon=False,
    fontsize=8.5,
    loc="upper right",
    title_fontsize=9,
    ncol=1,
)

# Reference wavelengths
for um, name in [(3.0, "L"), (10.0, "N"), (24.0, "MIPS-24"), (100.0, "FIR")]:
    ax.axvline(um * 1.0e4, color="0.85", lw=0.5, alpha=0.5)

fig.tight_layout()
plt.savefig("plot_skirtor_inclination_sweep.png", dpi=150, bbox_inches="tight")
print("Saved plot_skirtor_inclination_sweep.png")